In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from pathlib import Path
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import ParameterGrid
from sklearn.calibration import CalibrationDisplay
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss, roc_auc_score

In [13]:
MODEL_START_SEASON = 2010
VALIDATION_SEASON = 2024
TEST_SEASON = 2025
RANDOM_STATE = 42

In [14]:
df = pd.read_csv('../data/processed/model_dataset_v2.csv', parse_dates=['gameday'],)

In [15]:
df = (
    df
    .sort_values(['gameday', 'game_id'])
    .reset_index(drop=True)
)
df.shape

(4617, 80)

In [16]:
feature_columns = [
    'pregame_off_epa_blended',
    'pregame_off_success_rate_blended',
    'pregame_pass_epa_blended',
    'pregame_rush_epa_blended',
    'pregame_def_epa_allowed_blended',
    'pregame_def_success_rate_allowed_blended',
    'off_epa_last3',
    'off_success_rate_last3',
    'pass_epa_last3',
    'rush_epa_last3',
    'def_epa_allowed_last3',
    'def_success_rate_allowed_last3',
    'off_epa_last5',
    'off_success_rate_last5',
    'pass_epa_last5',
    'rush_epa_last5',
    'def_epa_allowed_last5',
    'def_success_rate_allowed_last5',
    'off_epa_ewm',
    'off_success_rate_ewm',
    'pass_epa_ewm',
    'rush_epa_ewm',
    'def_epa_allowed_ewm',
    'def_success_rate_allowed_ewm',
    'pregame_elo',
    'pregame_sos',
    'rest_days',
    'short_rest',
    'prior_games',
    'pregame_turnovers',
    'pregame_takeaways',
    'pregame_off_sack_rate',
    'pregame_def_sack_rate',
    'pregame_qb_epa',
    'pregame_qb_epa_last5',
    'pregame_qb_success_rate'
]

features = [
    f"{side}_{column}"
    for side in ["home", "away"]
    for column in feature_columns
] + ["neutral_site"]

target = 'home_win'

In [20]:
train = df[df['season'].between(MODEL_START_SEASON, 2023)].copy()
val = df[df['season'] == VALIDATION_SEASON].copy()
test = df[df['season'] == TEST_SEASON].copy()

X_train = train[features]
y_train = train[target]

X_val = val[features]
y_val = val[target]

X_test = test[features]
y_test = test[target]

print('Training Games:', len(train))
print('Validation Games:', len(val))
print('Test Games:', len(test))

Training Games: 3781
Validation Games: 285
Test Games: 284
